# Semana 10: Monitoramento de Aplicações, Logs e Observabilidade

## Módulo de Observabilidade e Operações — Fábrica Virtual Smart N1

Este notebook apresenta os conceitos de **Observabilidade**, os três pilares fundamentais (**Métricas, Logs e Traces**), a metodologia **RED/USE**, a arquitetura com **Prometheus e Grafana** e a análise automatizada de logs de aplicação.

### Objetivos de aprendizagem
- Compreender a evolução do Monitoramento tradicional para a **Observabilidade** em microsserviços.
- Dominar os 3 pilares da observabilidade: Métricas, Logs estruturados e Traces distribuídos (OpenTelemetry).
- Aplicar o padrão **RED** (*Rate, Errors, Duration*) e o padrão **USE** (*Utilization, Saturation, Errors*).
- Entender a arquitetura de amostragem por raspagem (*Pull-based*) do Prometheus e renderização visual no Grafana.
- Desenvolver um parser em Python para extração de métricas de performance a partir de logs estruturados em JSON.

---


## 1. Fundamentação Teórica

### 1.1 Os 3 Pilares da Observabilidade

```text
                          +-------------------------------+ 
                          |  PILARES DA OBSERVABILIDADE   |
                          +-------------------------------+ 
                                   /      |      \
                                  /       |       \
                                 v        v        v
                           +----------+ +------+ +----------+
                           | MÉTRICAS | | LOGS | |  TRACES  |
                           +----------+ +------+ +----------+
                                |          |          |
                                v          v          v
                          Prometheus /   Loki /   OpenTelemetry /
                           Grafana       ELK        Jaeger
```

- **Métricas (O QUÊ):** Valores numéricos agregados no tempo (ex: uso de CPU %, taxa de requisições/seg). Ideais para alertas e dashboards.
- **Logs (POR QUÊ):** Registros contendo contexto de um evento (ex: exceção com stacktrace). Ideais para depuração pontual.
- **Traces (ONDE):** Rastreamento do caminho percorrido por uma requisição através de múltiplos microsserviços (ID de correlação).

---

### 1.2 O Método RED (Rate, Errors, Duration)

Proposto por Tom Wilkie, é o padrão de monitoramento focado em serviços orientados a requisição (APIs):

1. **Rate (Taxa):** Número de requisições por segundo recebidas pela aplicação ($req/s$).
2. **Errors (Erros):** Quantidade ou percentual de requisições que falharam (respostas HTTP 5xx).
3. **Duration (Duração/Latência):** Tempo que as requisições levam para ser processadas (medido em percentis p50, p95 e p99).

---


## 2. Prática — Parser de Logs JSON e Extrator de Métricas RED em Python

Nesta prática, analisaremos uma massa de logs estruturados em formato JSON gerados pelos microsserviços da Smart N1 para calcular automaticamente a taxa de requisições, o percentual de erro HTTP e a latência média.

In [ ]:
import json

# Amostra de logs de aplicação estruturados em JSON
logs_brutos_json = [
    '{"timestamp": "2026-08-18T10:00:01Z", "method": "GET", "path": "/api/v1/telemetria", "status": 200, "latency_ms": 12.5}',
    '{"timestamp": "2026-08-18T10:00:02Z", "method": "POST", "path": "/api/v1/comando", "status": 200, "latency_ms": 45.0}',
    '{"timestamp": "2026-08-18T10:00:03Z", "method": "GET", "path": "/api/v1/telemetria", "status": 500, "latency_ms": 150.2}',
    '{"timestamp": "2026-08-18T10:00:04Z", "method": "GET", "path": "/api/v1/oee", "status": 200, "latency_ms": 18.1}',
    '{"timestamp": "2026-08-18T10:00:05Z", "method": "POST", "path": "/api/v1/comando", "status": 503, "latency_ms": 210.0}',
    '{"timestamp": "2026-08-18T10:00:06Z", "method": "GET", "path": "/api/v1/telemetria", "status": 200, "latency_ms": 14.0}',
]

def extrair_metricas_red(logs_json):
    total_requisicoes = len(logs_json)
    erros_5xx = 0
    latencias = []
    
    for linha in logs_json:
        dado = json.loads(linha)
        status = dado.get("status", 200)
        latencia = dado.get("latency_ms", 0.0)
        
        latencias.append(latencia)
        if status >= 500:
            erros_5xx += 1
            
    rate = total_requisicoes  # Total de requisições no lote
    taxa_erro_pct = round((erros_5xx / total_requisicoes) * 100.0, 1)
    latencia_media_ms = round(sum(latencias) / len(latencias), 1)
    
    return {
        "Rate (Total Requisições)": rate,
        "Errors (HTTP 5xx)": erros_5xx,
        "Error Rate (%)": f"{taxa_erro_pct}%",
        "Duration (Latência Média)": f"{latencia_media_ms} ms"
    }

metricas_red = extrair_metricas_red(logs_brutos_json)

print("=== MÉTRICAS RED EXTRAÍDAS DOS LOGS JSON ===\n")
for k, v in metricas_red.items():
    print(f"- {k}: {v}")


---

## 3. Exercícios de Fixação e Avaliação

### Questão 1
Qual a principal diferença entre o monitoramento tradicional baseado em ping/saúde de servidor e a **Observabilidade**? Por que a observabilidade é indispensável em arquiteturas de microsserviços orientadas a containers?

### Questão 2
Explique o funcionamento do método de amostragem por **Pull** utilizado pelo Prometheus (raspagem de métricas via endpoint `/metrics` HTTP) em comparação com o método de **Push** (envio direto pelo cliente).

### Questão 3
No contexto da análise de logs, por que o uso de **logs estruturados em JSON** é preferível ao uso de logs em texto puro (*unstructured text*) para sistemas de centralização como Grafana Loki ou ELK Stack?
